In [0]:
%sql
create schema if not exists workspace.gold

In [0]:
''' Aqui a coisa é simples, vou ler da prata, fazer os joins e uma logica para que eu nao pegue dados repetidos na soma da qunatidade total de faturamento. Pela logica utileizei ate o preço do frete para o calculo pois ele faz sentido esta no faturamento total'''
from pyspark.sql import functions as F

itens = spark.table("workspace.silver.fat_itens_pedidos")
pedidos = spark.table("workspace.silver.fat_pedido_total")
produtos = spark.table("workspace.silver.dim_produtos")

base_comercial = (
    itens
    .join(produtos, on="id_produto", how="left")
    .join(pedidos.select("id_pedido", "data_pedido", "valor_total_pago_brl","valor_total_pago_usd"), on="id_pedido", how="inner")
    .withColumn("receita_item_brl", F.col("preco_BRL") + F.col("preco_frete"))
    .withColumn("fator_usd", F.when(F.col("valor_total_pago_brl") > 0, F.col("valor_total_pago_usd") / F.col("valor_total_pago_brl")))
    .withColumn("receita_item_usd", F.col("receita_item_brl") * F.col("fator_usd"))
    .withColumn("ano_venda", F.year(F.col("data_pedido")))
    .withColumn("mes_venda", F.month(F.col("data_pedido")))
)

fat_vendas_comercial = (
    base_comercial
    .groupby("ano_venda", "mes_venda","categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.round(F.sum("receita_item_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("receita_item_usd"), 2).alias("receita_total_usd")
    )
    .withColumn("ticket_medio_brl", F.round(F.col("receita_total_brl")/F.col("total_pedidos"), 2))
    .orderBy("ano_venda", "mes_venda","categoria_produto")
)

fat_vendas_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fat_vendas_comercial")

AGORA eu vou pegar os rankings que a atividade pede.

In [0]:
#ranking de 5 produtos mais vendidos

ranking_produtos = (
    base_comercial.groupBy("nome_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
)

top5 = (
    ranking_produtos
    .orderBy(F.col("quantidade_vendida").desc(), F.col("nome_produto").asc()).limit(5)
)

display(top5)

nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


In [0]:
#os menos vendidos

ranking_produtos = (
    base_comercial.groupBy("nome_produto", "categoria_produto")
    .agg(F.count("id_item").alias("quantidade_vendida"))
)

top5 = (
    ranking_produtos
    .filter(F.col("quantidade_vendida")>0)
    .orderBy(F.col("quantidade_vendida").asc(), F.col("nome_produto").asc()).limit(5)
)

display(top5)

nome_produto,categoria_produto,quantidade_vendida
"""Smart TV 50"""" Básico""",eletronicos,1
"""Smart TV 50"""" Master""",eletronicos,1
Acessório Padrão,artigos_de_festas,1
Acessório Padrão,livros_importados,1
Acessório Padrão,fashion_roupa_infanto_juvenil,1


AGORA VAMOS PARA O PASSO 2 DA GOLD

In [0]:
""" Aqui eu fiz a logica para criar a tabela de avaliação para usar tanto no ranking de produtos como no ranking de vendedor"""
from pyspark.sql import functions as F

avaliacoes = spark.table("workspace.silver.fat_avaliacoes_pedidos")
itens = spark.table("workspace.silver.fat_itens_pedidos")
produtos = spark.table("workspace.silver.dim_produtos")
vendedores = spark.table("workspace.silver.dim_vendedores")

base_comercial = (
    avaliacoes
    .join(itens, on="id_pedido", how="inner")
    .join(produtos, on="id_produto", how="left")
    .join(vendedores, on="id_vendedor", how="left")
)

base_tabela = base_comercial.select(
    "id_pedido",
    "categoria_produto",
    "nome_vendedor",
    "estado",
    "nota_avaliacao"
).dropDuplicates()

fat_avaliacao_clientes = (
    base_tabela
    .groupBy("categoria_produto","nome_vendedor","estado")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.avg("nota_avaliacao").alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao")>=4,1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao")<=2,1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    .withColumn("percentual_satisfacao", F.col("total_avaliacoes_positivas")/F.col("total_avaliacoes")*100)
)

fat_avaliacao_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fat_avaliacoes_clientes")

AGORA VAMOS PARA OS RANKINGS DESSE SEGUNDO PASSO

In [0]:
#ranking de produtos

base_produtos_ranking = base_comercial.select(
        "id_pedido",
        "id_produto",
        "nome_produto",
        "categoria_produto",
        "nota_avaliacao"
    ).dropDuplicates()

produtos_ranking = (
    base_produtos_ranking
    .groupBy("id_produto","nome_produto", "categoria_produto")
    .agg(
        F.avg("nota_avaliacao").alias("avaliacao_media"),
        F.count("*").alias("total_avaliacoes")
    )
)

produtos_bem_avaliados = (
    produtos_ranking
    .orderBy(
        F.desc("avaliacao_media"),
        F.desc("total_avaliacoes")
    ).limit(1)
)

produto_menos_bem_avaliado = (
    produtos_ranking
    .orderBy(
        F.asc("avaliacao_media"),
        F.desc("total_avaliacoes")
    ).limit(1)
)

display(produtos_bem_avaliados)
display(produto_menos_bem_avaliado)


id_produto,nome_produto,categoria_produto,avaliacao_media,total_avaliacoes
2722b7e5f68e776d18fe901638034e54,Protetor Solar Branco,beleza_saude,5.0,12


id_produto,nome_produto,categoria_produto,avaliacao_media,total_avaliacoes
fb29f48bfea41db52e349454f433340e,Webcam Rosa,informatica_acessorios,1.0,6


In [0]:
#Ranking de vendedores

base_vendedores_ranking = base_comercial.select(
    "id_pedido",
    "id_vendedor",
    "nome_vendedor",
    "estado",
    "nota_avaliacao"
).dropDuplicates()

vendedores_ranking = (
    base_vendedores_ranking
    .groupBy("id_vendedor","nome_vendedor", "estado")
    .agg(
        F.avg("nota_avaliacao").alias("avaliacao_media"),
        F.count("*").alias("total_avaliacoes")
    )
)

vendedores_bem_avaliados = (
    vendedores_ranking
    .orderBy(
        F.desc("avaliacao_media"),
        F.desc("total_avaliacoes")
    ).limit(1)
)

vendedores_menos_bem_avaliados = (
    vendedores_ranking
    .orderBy(
        F.asc("avaliacao_media"),
        F.desc("total_avaliacoes")
    ).limit(1)
)

display(vendedores_bem_avaliados)
display(vendedores_menos_bem_avaliados)

id_vendedor,nome_vendedor,estado,avaliacao_media,total_avaliacoes
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,PR,5.0,31


id_vendedor,nome_vendedor,estado,avaliacao_media,total_avaliacoes
61f159ef6da2d441951d2c0efa719362,Josué Ribeiro,ES,1.0,3
